In [1]:
import json
import os
import pickle as pkl
import openjij as oj
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from dimod import BinaryQuadraticModel, ExactSolver
from dwave.samplers import SimulatedAnnealingSampler
from dwave.system import DWaveSampler, EmbeddingComposite
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from torch.utils.data import DataLoader, TensorDataset
from time import perf_counter

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import QuadraticMLP, LogisticRegression, SVM
from src.utils import build_sampler, evaluate
from src.training import train

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Idea

For the current parameter vector $w$, the batch loss is approximated locally by a second-order Taylor expansion:

$$
\mathcal{L}(w + \Delta) \approx \mathcal{L}(w) + g^\top \Delta + \frac{1}{2}\Delta^\top H\Delta,
$$

where $g = \nabla \mathcal{L}(w)$ and $H = \nabla^2 \mathcal{L}(w)$.

The annealer does not optimize the full continuous update directly. Instead, for a selected subset of parameters, it chooses binary variables $z_i \in \{0,1\}$ that encode the sign of a fixed-size step:

$$
\Delta_i = \eta(2z_i - 1),
$$

so $z_i = 1$ means a $+\eta$ step and $z_i = 0$ means a $-\eta$ step. Substituting this into the quadratic model turns the local optimization problem into a QUBO/BQM:

$$
E(z) = \sum_i a_i z_i + \sum_{i<j} b_{ij} z_i z_j + c.
$$

The annealer minimizes $E(z)$, then the proposed update is applied to the network parameters and accepted only if the true loss decreases.

## Loading sample Iris data for training

In [2]:
# Load Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

scaler = StandardScaler()
X = scaler.fit_transform(X)

X = torch.FloatTensor(X)
y = torch.LongTensor(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [3]:
training_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(training_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(training_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 105
Test samples: 45


## Training

In [4]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(4, [16, 8], 3)
classical_device = "cpu" # TODO: For now it is pointless to use cuda, because the bottleneck is not the classical computations but the sampling from BQM.

sampler, sampler_name = build_sampler(mode="simulated")
print(f"Using sampler: {sampler_name}")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

Using sampler: simulated


In [5]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = "quadratic-annealer-iris",
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 000 | train_loss=0.9072 | train_acc=0.762 | test_loss=0.9202 | test_acc=0.711 | 
Epoch 005 | train_loss=0.3759 | train_acc=0.971 | test_loss=0.4250 | test_acc=0.911 | 
Epoch 010 | train_loss=0.1555 | train_acc=0.971 | test_loss=0.2116 | test_acc=0.956 | 
Epoch 015 | train_loss=0.0877 | train_acc=0.981 | test_loss=0.1522 | test_acc=0.956 | 
Epoch 020 | train_loss=0.0652 | train_acc=0.981 | test_loss=0.0813 | test_acc=0.978 | 
Epoch 025 | train_loss=0.0589 | train_acc=0.981 | test_loss=0.0844 | test_acc=0.978 | 


2026/03/12 14:59:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 14:59:50 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 14:59:50 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 029 | train_loss=0.0528 | train_acc=0.981 | test_loss=0.0733 | test_acc=0.956 | 


# Classical optimization for benchmarking

In [6]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(4, [16, 8], 3)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Optimization using Adam optimizaer

In [7]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=30, 
    experiment_name = "adam-optimizer-iris",
)

Epoch 000 | train_loss=0.9320 | train_acc=0.476 | test_loss=0.9350 | test_acc=0.444 | 
Epoch 005 | train_loss=0.3920 | train_acc=0.867 | test_loss=0.4354 | test_acc=0.800 | 
Epoch 010 | train_loss=0.1928 | train_acc=0.971 | test_loss=0.2893 | test_acc=0.867 | 
Epoch 015 | train_loss=0.1021 | train_acc=0.971 | test_loss=0.1440 | test_acc=0.956 | 
Epoch 020 | train_loss=0.0675 | train_acc=0.981 | test_loss=0.1069 | test_acc=0.956 | 
Epoch 025 | train_loss=0.0487 | train_acc=1.000 | test_loss=0.1541 | test_acc=0.911 | 


2026/03/12 15:00:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 15:00:02 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 15:00:02 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 029 | train_loss=0.0430 | train_acc=0.990 | test_loss=0.1304 | test_acc=0.933 | 


## Optimization using LBFGS-style second-order optimizer

In [8]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = QuadraticMLP(4, [16, 8], 3)
classical_device = "cuda" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              )

In [9]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = "lbfgs-optimizer-iris",
)

Epoch 000 | train_loss=1.1882 | train_acc=0.343 | test_loss=1.2051 | test_acc=0.378 | 
Epoch 005 | train_loss=1.1880 | train_acc=0.343 | test_loss=1.2049 | test_acc=0.378 | 
Epoch 010 | train_loss=1.1879 | train_acc=0.343 | test_loss=1.2049 | test_acc=0.378 | 
Epoch 015 | train_loss=1.1879 | train_acc=0.343 | test_loss=1.2049 | test_acc=0.378 | 
Epoch 020 | train_loss=1.1879 | train_acc=0.343 | test_loss=1.2049 | test_acc=0.378 | 


2026/03/12 15:00:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 15:00:09 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 15:00:09 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 025 | train_loss=1.1879 | train_acc=0.343 | test_loss=1.2049 | test_acc=0.378 | 
Epoch 029 | train_loss=1.1879 | train_acc=0.343 | test_loss=1.2049 | test_acc=0.378 | 
